# 06 — Más Allá del Curso: Paradigmas Avanzados
**Algoritmos y Estructuras de Datos · Universidad de Talca**

---
> *"Este notebook es un mapa, no un destino. Muestra las rutas que existen más allá de lo que hemos recorrido."*

| Campo | Detalle |
|---|---|
| **Tópico** | S03 — Diseño de Algoritmos |
| **Notebook** | 06 de 06 — **CIERRE DEL BLOQUE** |
| **Duración estimada** | 45 minutos |
| **Prerequisito** | NB00–NB05 (panorama del bloque S03) |
| **Objetivo** | Motivación panorámica — NO profundizar, sino despertar curiosidad |

---
### ¿Qué veremos?

| Sección | Paradigma | Pregunta central |
|---|---|---|
| A | **Algoritmos de Aproximación** | ¿Y si el problema es NP-difícil? |
| B | **Búsqueda Local / Metaheurísticas** | ¿Y si no podemos explorar todo el espacio? |
| C | **Algoritmos Aleatorizados** | ¿Y si le damos un dado al algoritmo? |
| D | **¿Qué sigue?** | ¿Adónde llevan estos caminos? |

In [ ]:
# ── Verificación de dependencias ────────────────────────────────────────────
import sys

_reqs = {'numpy': 'numpy', 'matplotlib': 'matplotlib'}
_missing = []
for pkg, mod in _reqs.items():
    try:
        __import__(mod)
        print(f'✓ {pkg}')
    except ImportError:
        _missing.append(pkg)
        print(f'✗ {pkg} — falta')
if _missing:
    print(f'\nInstalar: !pip install {" ".join(_missing)}')
else:
    print(f'\nPython {sys.version.split()[0]} — listo.')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import HTML, display
import random, math
from typing import List, Tuple, Optional, Dict

# ── Paleta de colores del curso ─────────────────────────────────────────────
AZUL     = '#2196F3'
NARANJA  = '#FF9800'
VERDE    = '#4CAF50'
ROJO     = '#F44336'
MORADO   = '#9C27B0'
AZ_CLARO = '#90CAF9'
FONDO    = '#FAFAFA'
TEXTO    = '#212121'

print('✓ Librerías cargadas')

---
## Sección A — Algoritmos de Aproximación

### A.1 El problema: NP-difícil

Todos los paradigmas del curso (Greedy, DP, BT, B&B) buscan la solución **óptima**.  
Pero existe una clase de problemas —los **NP-difíciles**— para los que no se conoce ningún algoritmo eficiente que garantice el óptimo.

**Ejemplos de problemas NP-difíciles:**
- Viajante de Comercio (TSP): visitar n ciudades con distancia mínima
- Coloración de grafos: colorear un grafo con k colores
- Cobertura de vértices (Vertex Cover)
- Problema de satisfacibilidad (SAT)

Para estos problemas, la estrategia es diferente:

> **¿Qué pasa si en lugar del óptimo buscamos una solución que sea como máximo *α* veces el óptimo?**

Eso es un **algoritmo de aproximación con razón α**.

### A.2 Ejemplo: 2-Aproximación para Cobertura de Vértices

**Problema:** Dado un grafo G=(V,E), encuentra el subconjunto mínimo de vértices que "cubre" todas las aristas (cada arista tiene al menos un extremo en el conjunto).

**Algoritmo 2-aprox** (greedy, O(V+E)):
```
cobertura = {}
para cada arista (u,v) no cubierta:
    agregar u y v a la cobertura
    marcar todas las aristas de u y v como cubiertas
```

**¿Por qué da a lo más 2× el óptimo?**  
Cada arista que tomamos es "independiente" (ningún vértice de esta arista apareció antes). El óptimo debe incluir al menos uno de {u,v} para cubrir esa arista. Nosotros tomamos los dos → a lo más el doble.

In [ ]:
# ── 2-Aproximación para Vertex Cover ────────────────────────────────────────

def vertex_cover_2aprox(
        grafo: Dict[int, List[int]],
        verbose: bool = True
) -> List[int]:
    '''
    Algoritmo 2-aproximación para Minimum Vertex Cover.

    Garantía: |cobertura| <= 2 × |cobertura_óptima|

    Complejidad:
        Tiempo: O(V + E)
        Espacio: O(E)
    '''
    aristas_visitadas = set()
    cobertura = set()
    pasos = []

    for u in grafo:
        for v in grafo[u]:
            arista = (min(u, v), max(u, v))
            if arista not in aristas_visitadas:
                aristas_visitadas.add(arista)
                cobertura.add(u)
                cobertura.add(v)
                pasos.append((u, v, sorted(cobertura)))
                if verbose:
                    print(f'  Arista ({u},{v}) libre → agregar ambos. '
                          f'Cobertura: {sorted(cobertura)}')

    return sorted(cobertura)


# Grafo de ejemplo: 8 nodos
GRAFO_VC = {
    0: [1, 3],
    1: [0, 2, 4],
    2: [1, 5],
    3: [0, 4, 6],
    4: [1, 3, 5, 7],
    5: [2, 4],
    6: [3, 7],
    7: [4, 6],
}

print('Grafo de ejemplo (8 nodos):')
print('Ejecutando 2-aproximación para Vertex Cover...')
print()
cobertura_vc = vertex_cover_2aprox(GRAFO_VC)
print(f'\nCobertura 2-aprox : {cobertura_vc}')
print(f'Tamaño            : {len(cobertura_vc)}')
print('Óptimo real       : {4, 1} o equivalente → 4 nodos')
print(f'Razón             : {len(cobertura_vc)/4:.2f}x ≤ 2.0x  ✓')

In [ ]:
# ── Visualización: grafo con cobertura marcada ──────────────────────────────

def pos_circular(n: int) -> Dict[int, Tuple[float, float]]:
    return {i: (math.cos(2*math.pi*i/n), math.sin(2*math.pi*i/n)) for i in range(n)}


fig_vc, (ax_orig, ax_cob) = plt.subplots(1, 2, figsize=(13, 5))
fig_vc.patch.set_facecolor(FONDO)
fig_vc.suptitle('2-Aproximación para Vertex Cover', fontsize=13,
                fontweight='bold', color=TEXTO)

pos = pos_circular(len(GRAFO_VC))

for ax, titulo in [(ax_orig, 'Grafo original'), (ax_cob, 'Cobertura 2-aprox (verde)')]:
    ax.set_facecolor(FONDO)
    ax.axis('off')
    ax.set_xlim(-1.4, 1.4)
    ax.set_ylim(-1.4, 1.4)
    ax.set_title(titulo, fontsize=11, color=TEXTO)

    # Aristas
    dibujadas = set()
    for u, vecinos in GRAFO_VC.items():
        for v in vecinos:
            arista = (min(u, v), max(u, v))
            if arista not in dibujadas:
                dibujadas.add(arista)
                x0, y0 = pos[u]
                x1, y1 = pos[v]
                cubierta = (u in cobertura_vc or v in cobertura_vc) and ax == ax_cob
                lc = VERDE if cubierta else '#BDBDBD'
                lw = 2.0 if cubierta else 1.2
                ax.plot([x0, x1], [y0, y1], '-', color=lc, lw=lw, zorder=1, alpha=0.8)

    # Nodos
    for nid, (x, y) in pos.items():
        en_cobertura = (nid in cobertura_vc)
        if ax == ax_cob:
            color_nodo = VERDE if en_cobertura else AZ_CLARO
        else:
            color_nodo = AZUL
        circ = plt.Circle((x, y), 0.15, color=color_nodo, zorder=2)
        ax.add_patch(circ)
        ax.text(x, y, str(nid), ha='center', va='center',
                fontsize=12, fontweight='bold', color='white', zorder=3)

    if ax == ax_cob:
        n_arr = mpatches.Patch(color=AZ_CLARO, label='No en cobertura')
        c_arr = mpatches.Patch(color=VERDE,    label='En cobertura')
        ax.legend(handles=[c_arr, n_arr], loc='lower right', fontsize=9)

plt.tight_layout()
plt.show()

### A.3 ¿Qué significa ser 2-aproximado?

```
OPT = tamaño del óptimo real (desconocido, calcularlo sería NP-difícil)
ALG = tamaño de nuestra solución

Garantía: ALG ≤ 2 × OPT
```

En el ejemplo: OPT ≈ 4, ALG = 6 → 6 ≤ 2×4 = 8 ✓

**Conexión con lo visto en el curso:**  
> Es como Greedy, pero con una **garantía matemática** de qué tan lejos podemos estar del óptimo.

| Paradigma | Garantía de optimalidad |
|---|---|
| Greedy | Ninguna en general (puede fallar) |
| DP / BT / B&B | 100% óptimo (pero puede ser exponencial) |
| **Aproximación** | **A lo más α × OPT en tiempo polinomial** |

---
## Sección B — Búsqueda Local y Metaheurísticas

### B.1 El problema: espacios enormes con muchos óptimos locales

Imagina optimizar una función con **cientos de máximos locales**. Backtracking exploraría todo el espacio, lo que es impracticable. Las metaheurísticas exploran de forma inteligente sin garantizar el óptimo, pero funcionan muy bien en la práctica.

### B.2 Hill Climbing

```
solución = inicio_aleatorio()
mientras haya mejora:
    vecino = mejor_vecino(solución)
    si f(vecino) > f(solución):
        solución = vecino
    sino:
        parar  ← ¡atrapado en óptimo local!
```

**Problema:** se queda atascado en el primer óptimo local que encuentra.

### B.3 Simulated Annealing (Recocido Simulado)

Inspirado en el recocido del metal: al enfriar metal lentamente, los átomos encuentran una configuración de energía mínima.

```
T = temperatura_inicial   ← controla la aleatoriedad
solución = inicio_aleatorio()
mientras T > T_final:
    vecino = vecino_aleatorio(solución)
    Δ = f(vecino) - f(solución)
    si Δ > 0:                           ← mejora: siempre acepta
        solución = vecino
    sino con prob. exp(Δ/T):            ← empeora: acepta con probabilidad
        solución = vecino
    T = T × α                           ← enfriamiento gradual
```

La clave: al principio (T alto) acepta soluciones peores → escapa de óptimos locales.  
Al final (T bajo) se vuelve más selectivo → converge a una buena solución.

In [ ]:
# ── Función multimodal para demostración ────────────────────────────────────

def f_multimodal(x: float) -> float:
    '''Función con varios máximos locales y un máximo global.'''
    return (np.sin(x) + np.sin(2.7 * x) + np.sin(0.5 * x)
            + 0.1 * np.sin(10 * x))


def hill_climbing(
        f, x_init: float, paso: float = 0.05,
        x_min: float = 0.0, x_max: float = 10.0,
        max_iter: int = 1000
) -> Tuple[float, float, List[Tuple[float, float]]]:
    '''
    Hill Climbing: busca máximo local moviendo +/- paso.

    Complejidad:
        Tiempo: O(max_iter)
        Espacio: O(max_iter) historial
    '''
    x = x_init
    historial = [(x, f(x))]
    for _ in range(max_iter):
        candidatos = []
        for delta in [-paso, paso]:
            nx = x + delta
            if x_min <= nx <= x_max:
                candidatos.append((f(nx), nx))
        if not candidatos:
            break
        mejor_f, mejor_x = max(candidatos)
        if mejor_f <= f(x):
            break
        x = mejor_x
        historial.append((x, f(x)))
    return x, f(x), historial


def simulated_annealing(
        f, x_init: float, T0: float = 5.0, T_fin: float = 0.01,
        alpha: float = 0.995, paso: float = 0.5,
        x_min: float = 0.0, x_max: float = 10.0,
        seed: int = 42
) -> Tuple[float, float, List[Tuple[float, float]]]:
    '''
    Simulated Annealing: acepta soluciones peores con probabilidad exp(Δ/T).

    Complejidad:
        Tiempo: O(log(T0/T_fin) / log(1/alpha)) iteraciones
        Espacio: O(iteraciones) historial
    '''
    rng = random.Random(seed)
    x = x_init
    mejor_x = x
    mejor_f = f(x)
    T = T0
    historial = [(x, f(x))]
    while T > T_fin:
        delta_x = rng.uniform(-paso, paso)
        nx = max(x_min, min(x_max, x + delta_x))
        delta_f = f(nx) - f(x)
        if delta_f > 0:
            x = nx
        elif rng.random() < math.exp(delta_f / T):
            x = nx
        if f(x) > mejor_f:
            mejor_f = f(x)
            mejor_x = x
        historial.append((x, f(x)))
        T *= alpha
    return mejor_x, mejor_f, historial


# Probar con inicio en el mismo punto
X_INIT = 0.5
x_hc, f_hc, hist_hc = hill_climbing(f_multimodal, X_INIT)
x_sa, f_sa, hist_sa = simulated_annealing(f_multimodal, X_INIT)

# Óptimo real por búsqueda densa
xs_grid = np.linspace(0, 10, 10000)
idx_opt = np.argmax([f_multimodal(x) for x in xs_grid])
x_opt_real = xs_grid[idx_opt]
f_opt_real = f_multimodal(x_opt_real)

print(f'Inicio         : x={X_INIT:.2f}, f={f_multimodal(X_INIT):.4f}')
print(f'Hill Climbing  : x={x_hc:.4f}, f={f_hc:.4f}  (pasos: {len(hist_hc)})')
print(f'Simul. Anneal. : x={x_sa:.4f}, f={f_sa:.4f}  (pasos: {len(hist_sa)})')
print(f'Óptimo real    : x={x_opt_real:.4f}, f={f_opt_real:.4f}')
print()
gap_hc = (f_opt_real - f_hc) / abs(f_opt_real) * 100
gap_sa = (f_opt_real - f_sa) / abs(f_opt_real) * 100
print(f'Gap HC  vs óptimo: {gap_hc:.2f}%')
print(f'Gap SA  vs óptimo: {gap_sa:.2f}%')

In [ ]:
# ── Hill Climbing vs Simulated Annealing, en texto ──────────────────────────
import random, math

def hill_climbing(f, x0, paso=0.1, iteraciones=200):
    """Solo acepta movimientos que mejoran. Se queda en el primer óptimo local."""
    x = x0
    for _ in range(iteraciones):
        cand = x + random.uniform(-paso, paso)
        if f(cand) > f(x):
            x = cand
    return x


def simulated_annealing(f, x0, paso=0.1, iteraciones=200, t0=2.0):
    """Acepta empeorar con probabilidad decreciente: puede salir de un óptimo local."""
    x = x0
    for k in range(iteraciones):
        T = t0 * (1 - k / iteraciones) + 1e-9
        cand = x + random.uniform(-paso, paso)
        delta = f(cand) - f(x)
        if delta > 0 or random.random() < math.exp(delta / T):
            x = cand
    return x


random.seed(7)
print(f"{'inicio':>8} {'hill climbing':>16} {'simulated annealing':>22}")
print("-" * 50)
for x0 in (-2.0, -0.5, 0.5, 2.0):
    xh = hill_climbing(f_multimodal, x0)
    xs = simulated_annealing(f_multimodal, x0)
    print(f"{x0:>8.1f} {f_multimodal(xh):>16.4f} {f_multimodal(xs):>22.4f}")

print("\n👉 Hill Climbing depende fuertemente del punto de partida: se queda en el")
print("   óptimo LOCAL más cercano. Simulated Annealing acepta empeorar al principio,")
print("   cuando la temperatura es alta, y por eso escapa de esos óptimos locales.")

In [ ]:
# ── El efecto de la temperatura inicial: parametrizable ─────────────────────
# Cambia los valores de t0 y vuelve a ejecutar.
import random

def explorar_temperatura(temperaturas=(0.01, 0.1, 1.0, 5.0), repeticiones=30):
    """Promedia el resultado de simulated annealing para distintas temperaturas."""
    print(f"{'T inicial':>11} {'valor medio':>14}  interpretación")
    print("-" * 62)
    for t0 in temperaturas:
        random.seed(7)
        vals = [f_multimodal(simulated_annealing(f_multimodal, random.uniform(-3, 3), t0=t0))
                for _ in range(repeticiones)]
        media = sum(vals) / len(vals)
        if t0 <= 0.05:
            nota = "casi hill climbing: no escapa de óptimos locales"
        elif t0 >= 5:
            nota = "demasiado caliente: acepta casi todo, camina al azar"
        else:
            nota = "equilibrio entre explorar y explotar"
        print(f"{t0:>11.2f} {media:>14.4f}  {nota}")


explorar_temperatura()
print("\n👉 La temperatura es el mando entre EXPLORAR (aceptar empeorar, buscar lejos)")
print("   y EXPLOTAR (solo mejorar, afinar donde estoy). Ni muy fría ni muy caliente.")

### B.4 Conexión con lo visto en el curso

> Simulated Annealing es como **Backtracking aleatorizado**: en lugar de explorar sistemáticamente todo el árbol, salta por el espacio de soluciones con cierta dosis de aleatoriedad controlada.

| Paradigma | Garantía | Velocidad | Úsalo cuando |
|---|---|---|---|
| Backtracking | Óptimo garantizado | Exponencial | n pequeño, necesitas exactitud |
| Hill Climbing | Solo óptimo local | O(iter) | Función bien comportada, sin muchos óptimos locales |
| **Simulated Annealing** | **Buena solución en práctica** | **O(iter)** | **n grande, calidad práctica > exactitud** |

**Aplicaciones reales de SA:**
- Diseño de chips VLSI (colocar millones de transistores)
- Planificación de rutas de aviones
- Alineamiento de secuencias genéticas

---
## Sección C — Algoritmos Aleatorizados

### C.1 Dos categorías

| Categoría | Tiempo | Resultado | Ejemplo |
|---|---|---|---|
| **Las Vegas** | Aleatorio | **Siempre correcto** | QuickSort con pivot aleatorio |
| **Monte Carlo** | Fijo | Correcto con alta probabilidad | Estimación de π |

### C.2 Las Vegas — QuickSort aleatorizado

QuickSort determinístico: si el pivot siempre cae en el extremo → O(n²).
QuickSort aleatorizado: pivot al azar → O(n log n) **esperado** en cualquier entrada.

**Conexión con D&V:** es Divide y Vencerás, pero el punto de división es aleatorio.  
El resultado es **siempre correcto** (la lista siempre queda ordenada). Solo el tiempo varía.

In [ ]:
# ── QuickSort aleatorizado (Las Vegas) ──────────────────────────────────────

def quicksort_det(arr: List[int]) -> Tuple[List[int], int]:
    '''
    QuickSort determinístico (pivot = último elemento).
    Retorna (lista_ordenada, comparaciones).

    Complejidad:
        Tiempo: O(n²) peor caso (entrada ordenada), O(n log n) promedio
        Espacio: O(n) recursión
    '''
    comps = [0]

    def _qs(a: List[int]) -> List[int]:
        if len(a) <= 1:
            return a
        pivot = a[-1]    # siempre el último
        izq, der = [], []
        for x in a[:-1]:
            comps[0] += 1
            (izq if x <= pivot else der).append(x)
        return _qs(izq) + [pivot] + _qs(der)

    return _qs(arr[:]), comps[0]


def quicksort_rand(arr: List[int], seed: int = None) -> Tuple[List[int], int]:
    '''
    QuickSort aleatorizado (pivot uniformemente al azar).
    Retorna (lista_ordenada, comparaciones).

    Complejidad:
        Tiempo: O(n log n) esperado en CUALQUIER entrada
        Espacio: O(n) recursión
    '''
    rng = random.Random(seed)
    comps = [0]

    def _qs(a: List[int]) -> List[int]:
        if len(a) <= 1:
            return a
        idx = rng.randint(0, len(a) - 1)
        pivot = a[idx]
        resto = a[:idx] + a[idx+1:]
        izq, der = [], []
        for x in resto:
            comps[0] += 1
            (izq if x <= pivot else der).append(x)
        return _qs(izq) + [pivot] + _qs(der)

    return _qs(arr[:]), comps[0]


# Peor caso para QS determinístico: arreglo ya ordenado
n_test = 200
arr_peor = list(range(n_test))          # ordenado = peor para det
arr_rand = list(range(n_test))
random.Random(7).shuffle(arr_rand)      # aleatorio

_, comp_det_peor = quicksort_det(arr_peor)
_, comp_rand_peor = quicksort_rand(arr_peor, seed=42)
_, comp_det_norm = quicksort_det(arr_rand)
_, comp_rand_norm = quicksort_rand(arr_rand, seed=42)

print(f'n = {n_test}')
print(f'{"Entrada":<25} {"QS Det":>10} {"QS Rand":>10}')
print('-' * 47)
print(f'{"Ordenado (peor caso)":<25} {comp_det_peor:>10,} {comp_rand_peor:>10,}')
print(f'{"Aleatorio (caso normal)":<25} {comp_det_norm:>10,} {comp_rand_norm:>10,}')
print()
ref_nlogn = int(n_test * math.log2(n_test))
ref_n2    = n_test ** 2
print(f'Referencia n log n ≈ {ref_nlogn:,}')
print(f'Referencia n²      = {ref_n2:,}')
print()
print('→ QS aleatorizado evita el peor caso O(n²) sin importar la entrada.')

In [ ]:
# ── Monte Carlo — Estimación de π ───────────────────────────────────────────

def estimar_pi_montecarlo(
        n_puntos: int, seed: int = 42
) -> Tuple[float, List[float]]:
    '''
    Estima π usando el método de Monte Carlo.

    Idea: puntos aleatorios en cuadrado [0,1]×[0,1].
    Proporción dentro del círculo unitario ≈ π/4.

    Complejidad:
        Tiempo: O(n)
        Espacio: O(n) historial
    Error esperado: O(1/√n) — baja muy lentamente
    '''
    rng = random.Random(seed)
    dentro = 0
    estimaciones = []
    for i in range(1, n_puntos + 1):
        x = rng.random()
        y = rng.random()
        if x*x + y*y <= 1.0:
            dentro += 1
        estimaciones.append(4 * dentro / i)
    return estimaciones[-1], estimaciones


N_MC = 5000
pi_est, hist_pi = estimar_pi_montecarlo(N_MC)

fig_mc, (ax_mc1, ax_mc2) = plt.subplots(1, 2, figsize=(13, 5))
fig_mc.patch.set_facecolor(FONDO)
fig_mc.suptitle(f'Monte Carlo — Estimación de π ({N_MC} puntos)',
                fontsize=13, fontweight='bold', color=TEXTO)

# Panel 1: puntos en el cuadrado
rng_vis = random.Random(42)
N_VIS = 500
xs_mc = [rng_vis.random() for _ in range(N_VIS)]
ys_mc = [rng_vis.random() for _ in range(N_VIS)]
colores_mc = [AZUL if (x**2 + y**2 <= 1) else NARANJA
              for x, y in zip(xs_mc, ys_mc)]

ax_mc1.set_facecolor(FONDO)
ax_mc1.scatter(xs_mc, ys_mc, c=colores_mc, s=6, alpha=0.7, zorder=2)
theta = np.linspace(0, math.pi/2, 200)
ax_mc1.plot(np.cos(theta), np.sin(theta), '-', color=ROJO, lw=2, zorder=3,
            label='Arco (radio=1)')
dentro_vis = sum(1 for c in colores_mc if c == AZUL)
pi_aprox_vis = 4 * dentro_vis / N_VIS
ax_mc1.set_title(f'n={N_VIS}: π ≈ {pi_aprox_vis:.4f}', fontsize=11, color=TEXTO)
ax_mc1.set_aspect('equal')
ax_mc1.tick_params(colors=TEXTO)
p1 = mpatches.Patch(color=AZUL,   label='Dentro del círculo')
p2 = mpatches.Patch(color=NARANJA, label='Fuera del círculo')
ax_mc1.legend(handles=[p1, p2], fontsize=8)

# Panel 2: convergencia
ax_mc2.set_facecolor(FONDO)
ax_mc2.plot(range(1, N_MC + 1), hist_pi, '-', color=AZUL, lw=1.2, alpha=0.8,
            label='Estimación π')
ax_mc2.axhline(math.pi, color=ROJO, lw=2, ls='--', label=f'π real = {math.pi:.6f}')
ax_mc2.set_xlabel('Número de puntos', color=TEXTO)
ax_mc2.set_ylabel('Estimación de π', color=TEXTO)
ax_mc2.set_title(f'Convergencia ({N_MC} muestras) → π ≈ {pi_est:.6f}',
                 fontsize=11, color=TEXTO)
ax_mc2.legend(fontsize=9)
ax_mc2.tick_params(colors=TEXTO)
ax_mc2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

error_pct = abs(pi_est - math.pi) / math.pi * 100
print(f'Error con n={N_MC}: {error_pct:.4f}%')
print(f'Para reducir el error a la mitad necesitaríamos n={N_MC * 4} puntos (error ∝ 1/√n).')

In [ ]:
# ── Comparación: QS det vs QS rand — comparaciones por tamaño ─────────────

ns_qs = [50, 100, 200, 400, 800]
comps_det_peor_l, comps_rand_peor_l = [], []
comps_det_rand_l, comps_rand_rand_l = [], []

for n_i in ns_qs:
    a_peor = list(range(n_i))
    a_al   = list(range(n_i))
    random.Random(7).shuffle(a_al)
    _, cd_p  = quicksort_det(a_peor)
    _, cr_p  = quicksort_rand(a_peor, seed=42)
    _, cd_al = quicksort_det(a_al)
    _, cr_al = quicksort_rand(a_al, seed=42)
    comps_det_peor_l.append(cd_p)
    comps_rand_peor_l.append(cr_p)
    comps_det_rand_l.append(cd_al)
    comps_rand_rand_l.append(cr_al)

fig_qs, ax_qs = plt.subplots(figsize=(10, 5))
fig_qs.patch.set_facecolor(FONDO)
ax_qs.set_facecolor(FONDO)

ax_qs.plot(ns_qs, comps_det_peor_l,  'o--', color=ROJO,    lw=2, ms=7,
           label='QS Det — entrada ordenada (peor caso)')
ax_qs.plot(ns_qs, comps_rand_peor_l, 's-',  color=NARANJA, lw=2, ms=7,
           label='QS Rand — entrada ordenada')
ax_qs.plot(ns_qs, comps_det_rand_l,  'D--', color=AZ_CLARO, lw=2, ms=7,
           label='QS Det — entrada aleatoria')
ax_qs.plot(ns_qs, comps_rand_rand_l, '^-',  color=AZUL,    lw=2, ms=7,
           label='QS Rand — entrada aleatoria')

ref_n2    = [n**2 for n in ns_qs]
ref_nlogn = [int(n * math.log2(n)) for n in ns_qs]
ax_qs.plot(ns_qs, ref_n2,    ':', color='#9E9E9E', lw=1.5, label='n² (referencia)')
ax_qs.plot(ns_qs, ref_nlogn, ':', color='#616161', lw=1.5, label='n log n (referencia)')

ax_qs.set_xlabel('Tamaño n', color=TEXTO, fontsize=11)
ax_qs.set_ylabel('Comparaciones', color=TEXTO, fontsize=11)
ax_qs.set_title('QuickSort Determinístico vs Aleatorizado',
                fontsize=12, color=TEXTO)
ax_qs.legend(fontsize=8, loc='upper left')
ax_qs.tick_params(colors=TEXTO)
ax_qs.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### C.3 Las Vegas vs Monte Carlo — Resumen

| | Las Vegas | Monte Carlo |
|---|---|---|
| **Resultado** | Siempre correcto | Correcto con prob. p |
| **Tiempo** | Aleatorio (esperado acotado) | Determinístico |
| **Cuándo falla** | Tarda mucho (rara vez) | Da resultado incorrecto (con prob. pequeña) |
| **Ejemplo** | QuickSort aleatorizado | Estimación de π, tests de primalidad |
| **Reducir error** | Ejecutar de nuevo si tarda | Aumentar n (error ∝ 1/√n) |

> **Conexión con el curso:** QuickSort aleatorizado es Divide y Vencerás con punto de división aleatorio. La aleatorización *rompe la adversarialidad*: ninguna entrada puede ser siempre el peor caso.

---
## Sección D — ¿Qué sigue?

### D.1 Panorama completo de paradigmas

| Paradigma | Idea central | Garantía | Complejidad típica | Dónde aprender más |
|---|---|---|---|---|
| Divide y Vencerás | Dividir → Vencer → Combinar | Óptimo | O(n log n) | CLRS Cap. 4, Kleinberg Cap. 5 |
| Algoritmos Voraces | Mejor decisión local en cada paso | Solo si se puede probar | O(n log n) | CLRS Cap. 15, Kleinberg Cap. 4 |
| Programación Dinámica | Tabla de subproblemas solapados | Óptimo | O(n·W) o O(n²) | CLRS Cap. 14, AtCoder DP Contest |
| Backtracking | Búsqueda exhaustiva con retroceso | Óptimo | O(2ⁿ) o O(n!) | Skiena Cap. 9 |
| Branch & Bound | BT + cota superior/inferior | Óptimo | O(2ⁿ) práct. menor | Skiena Cap. 9 |
| **Aprox.** | Solución α-óptima en tiempo polinomial | α × OPT | Polinomial | CLRS Cap. 35 |
| **Metaheurísticas** | Búsqueda aleatoria inteligente | Buena solución práctica | O(iter) | Nocedal & Wright |
| **Aleatorizados** | Aleatorizar para romper casos adversariales | Correcto (LV) / prob. (MC) | O(n log n) esp. | MitzenmacherUpfal |
| **Prog. Lineal Entera** | Optimización continua/discreta | Óptimo (si LP factible) | Exp. peor, polinomial práct. | Cormen Apéndice, Bertsimas |

### D.2 Cursos donde profundizar

| Curso (típico) | Paradigmas que profundiza |
|---|---|
| **Diseño y Análisis de Algoritmos** (postgrado) | Aproximación, Aleatorizados, Complejidad NP |
| **Investigación de Operaciones** | Programación Lineal Entera, B&B avanzado |
| **Inteligencia Artificial** | Metaheurísticas, Búsqueda heurística (A*), RL |
| **Machine Learning** | Optimización estocástica (SGD, Adam) |
| **Computación de Alto Rendimiento** | Algoritmos paralelos y distribuidos |

In [ ]:
# ── Mapa visual de conexiones entre paradigmas ──────────────────────────────

fig_map, ax_map = plt.subplots(figsize=(14, 8))
fig_map.patch.set_facecolor(FONDO)
ax_map.set_facecolor(FONDO)
ax_map.axis('off')
ax_map.set_title('Mapa de Paradigmas Algorítmicos — S03 Diseño de Algoritmos',
                  fontsize=13, fontweight='bold', color=TEXTO, pad=15)

# Nodos: (x, y, label, color, estilo)
nodos_map = [
    # Curso
    (0.50, 0.90, 'DISEÑO\nDE ALGORITMOS', TEXTO,     'round,pad=0.5', 14),
    (0.15, 0.65, 'Divide y\nVencerás',   AZUL,      'round,pad=0.4', 10),
    (0.38, 0.65, 'Greedy',               NARANJA,    'round,pad=0.4', 10),
    (0.62, 0.65, 'Prog.\nDinámica',      MORADO,     'round,pad=0.4', 10),
    (0.85, 0.65, 'Backtracking',         VERDE,      'round,pad=0.4', 10),
    (0.85, 0.42, 'Branch &\nBound',      VERDE,      'round,pad=0.4', 10),
    # Avanzados
    (0.15, 0.22, 'Algoritmos\nAleatorizados', AZ_CLARO, 'round,pad=0.4', 10),
    (0.40, 0.22, 'Aproximación',         NARANJA,    'round,pad=0.4', 10),
    (0.65, 0.22, 'Metaheurísticas',      ROJO,       'round,pad=0.4', 10),
    (0.87, 0.22, 'Prog. Lineal\nEntera', MORADO,     'round,pad=0.4', 10),
]

# Aristas (índice_origen, índice_destino, etiqueta_arista)
aristas_map = [
    (0, 1, ''),
    (0, 2, ''),
    (0, 3, ''),
    (0, 4, ''),
    (4, 5, 'Añade cota'),
    (1, 6, 'aleatoriza'),
    (2, 7, 'garantía\nα×OPT'),
    (4, 8, 'aleatoriza'),
    (5, 9, 'continuo'),
]

label_curso = ['Divide y Vencerás', 'Greedy', 'Prog. Dinámica',
               'Backtracking', 'Branch & Bound']
label_avanz = ['Aleatorizados', 'Aproximación', 'Metaheurísticas', 'Prog. Lineal Entera']

# Dibujar aristas
for i, j, lbl in aristas_map:
    x0, y0 = nodos_map[i][0], nodos_map[i][1]
    x1, y1 = nodos_map[j][0], nodos_map[j][1]
    ax_map.annotate('', xy=(x1, y1), xytext=(x0, y0),
                    arrowprops=dict(arrowstyle='->', color='#9E9E9E',
                                   lw=1.5, connectionstyle='arc3,rad=0.0'),
                    xycoords='axes fraction', textcoords='axes fraction')
    if lbl:
        xm, ym = (x0 + x1) / 2, (y0 + y1) / 2
        ax_map.text(xm + 0.01, ym, lbl, transform=ax_map.transAxes,
                    fontsize=7, color='#616161', ha='center', va='center',
                    style='italic')

# Dibujar nodos
for x, y, lbl, color, box, fs in nodos_map:
    fc = 'white' if color == TEXTO else color
    tc = TEXTO if color == TEXTO else 'white'
    ax_map.text(x, y, lbl, transform=ax_map.transAxes,
                ha='center', va='center', fontsize=fs,
                color=tc, fontweight='bold',
                bbox=dict(boxstyle=box, facecolor=fc,
                          edgecolor=color, linewidth=2, alpha=0.92),
                zorder=3)

# Etiquetas de zona
ax_map.text(0.01, 0.50, 'EN ESTE\nCURSO',
            transform=ax_map.transAxes, fontsize=9,
            color='#388E3C', fontweight='bold', va='center',
            rotation=90)
ax_map.text(0.01, 0.15, 'SIGUIENTES\nCURSOS',
            transform=ax_map.transAxes, fontsize=9,
            color='#1565C0', fontweight='bold', va='center',
            rotation=90)

ax_map.axhline(0.37, color='#E0E0E0', lw=1.5, ls='--', alpha=0.8)
ax_map.set_xlim(0, 1)
ax_map.set_ylim(0.05, 1.0)

plt.tight_layout()
plt.show()

In [ ]:
# ── Resumen: dónde encaja cada paradigma avanzado ───────────────────────────
filas = [
    ["Aproximación",      "NP-difícil, quiero garantía",  "Voraz + demostración del error",  "factor conocido del óptimo"],
    ["Metaheurística",    "Espacio enorme, sin garantía", "Backtracking + azar",             "sin garantía, suele bastar"],
    ["Las Vegas",         "Quiero respuesta SIEMPRE correcta", "Divide y vencerás + azar",   "tiempo aleatorio"],
    ["Monte Carlo",       "Puedo tolerar error pequeño",  "Muestreo aleatorio",              "correcto con probabilidad alta"],
    ["Prog. lineal entera","Restricciones lineales",      "Ramificación y poda + LP",        "óptimo exacto, costo alto"],
]
cols = ["Paradigma", "Cuándo aparece", "Se parece a", "Qué garantiza"]
anchos = [22, 32, 34, 30]

print("".join(c.ljust(w) for c, w in zip(cols, anchos)))
print("-" * sum(anchos))
for f in filas:
    print("".join(str(v).ljust(w) for v, w in zip(f, anchos)))

print("\n👉 Todos EXTIENDEN los cuatro paradigmas fundamentales de la semana 2;")
print("   ninguno los reemplaza. Domina los cuatro básicos y estos se leen solos.")

---
## Autoevaluación final del bloque S03

## ✍️ Autoevaluación

Repasa los paradigmas avanzados contrastándolos con los cuatro fundamentales de la semana 2:

1. ¿En qué se parece una metaheurística a un algoritmo voraz, y en qué se diferencia?
2. ¿Qué garantiza un algoritmo de aproximación que una metaheurística no garantiza?
3. ¿Por qué ramificación y poda necesita una cota y el backtracking simple no?


---
## Lecturas recomendadas

### Algoritmos de Aproximación

| Recurso | Capítulo |
|---|---|
| **CLRS** — *Introduction to Algorithms* (4ª ed.) | Cap. 35 — Approximation Algorithms |
| **Vazirani** — *Approximation Algorithms* (Springer) | Cap. 1–3 (gratuito en muchas bibliotecas) |

### Metaheurísticas

| Recurso | Contenido |
|---|---|
| **Kirkpatrick et al. (1983)** — *Science* | Artículo original de Simulated Annealing |
| **Russell & Norvig** — *Artificial Intelligence* (4ª ed.) | Cap. 4 — Local Search Algorithms |

### Algoritmos Aleatorizados

| Recurso | Capítulo |
|---|---|
| **Mitzenmacher & Upfal** — *Probability and Computing* (2ª ed.) | Cap. 1–3 (excelente introducción) |
| **Motwani & Raghavan** — *Randomized Algorithms* | Cap. 1 |

### Recursos online

- **VisuAlgo** — [visualgo.net/en](https://visualgo.net/en): visualizaciones interactivas
- **CP-Algorithms** — [cp-algorithms.com](https://cp-algorithms.com): algoritmos con código
- **USACO Guide** — [usaco.guide](https://usaco.guide): currículo estructurado para competitive programming
- **AtCoder Educational DP Contest** — [atcoder.jp/contests/dp](https://atcoder.jp/contests/dp): 26 problemas DP graduados

---
## Cierre del Bloque S03

```
S03 — Diseño de Algoritmos
├── NB00: Introducción y panorama comparativo
├── NB01: Divide y Vencerás     → findMax, Binary Search, Merge Sort
├── NB02: Greedy                → Mochila*, Huffman, Dijkstra
├── NB03: Programación Dinámica → Mochila* (óptimo), Fibonacci, Levenshtein
├── NB04: Backtracking          → Mochila* (árbol completo), N-Reinas
├── NB05: Branch & Bound        → Mochila* (árbol podado), 8-Puzzle
└── NB06: Más allá              → Aproximación, Metaheurísticas, Aleatorizados
         ↑
         * mismos 5 objetos del PDF en los 4 notebooks → comparación directa
```

> **Los 5 objetos del PDF recorrieron 4 paradigmas y demostraron en la práctica por qué no hay un solo algoritmo para todo: cada paradigma brilla en su contexto.**